## 01 — GeoJSON Structure

In the previous lesson we loaded plain JSON and navigated it as Python dicts and lists.

**GeoJSON** is still just JSON — the same rules apply. The difference is that GeoJSON follows a specific convention about what the keys mean. Once you learn that convention, every GeoJSON file you encounter will look familiar.

The example file for this lesson is `data/meteorites.geojson` — a dataset of meteorite landing sites around the world.

## The GeoJSON Shape

Every valid GeoJSON file is built from three nested layers:

```
FeatureCollection          ← the whole file
└── features: [ ... ]      ← a list of Feature objects
    └── Feature            ← one geographic thing
        ├── type: "Feature"
        ├── geometry       ← WHERE it is
        │   ├── type: "Point"
        │   └── coordinates: [lon, lat]
        └── properties     ← WHAT it is
            ├── name: "Aarhus"
            ├── mass: 720
            └── year: 1951
```

**Critical detail: coordinates are `[longitude, latitude]`, not `[lat, lon]`.**  
This is backwards from how most people say it out loud. It trips up everyone the first time.

## Loading the File

In [1]:
import json
from pathlib import Path

path = Path("data/meteorites.geojson")
data = json.loads(path.read_text())

# inspect the top-level structure
print(type(data))           # <class 'dict'>
print(data.keys())          # dict_keys(['type', 'features'])
print(data["type"])         # FeatureCollection
print(len(data["features"])) # number of meteorite records

<class 'dict'>
dict_keys(['type', 'features'])
FeatureCollection
31963


## Accessing a Single Feature

Unlike `latex_colors.json` where the top level was a list, here `data` is a dict and the list lives at `data["features"]`.

In [2]:
feature = data["features"][0]

print(feature["type"])                        # Feature
print(feature["properties"]["name"])          # Aarhus
print(feature["properties"]["mass"])          # 720
print(feature["properties"]["year"])          # 1951
print(feature["geometry"]["type"])            # Point
print(feature["geometry"]["coordinates"])     # [10.23333, 56.18333]

Feature
Aarhus
720
1951
Point
[10.23333, 56.18333]


## Unpacking Coordinates

`coordinates` is an array: `[longitude, latitude]`.  
Index `0` is longitude (east/west), index `1` is latitude (north/south).

In [3]:
coords = feature["geometry"]["coordinates"]

lon = coords[0]   # 10.23333
lat = coords[1]   # 56.18333

print(f"lon: {lon},  lat: {lat}")

lon: 10.23333,  lat: 56.18333


## Traversal — Loop Over All Features

In [4]:
for feature in data["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]
    print(f"{props['name']:<30} mass: {props['mass']:>10}  lon: {coords[0]}")

Aarhus                         mass:        720  lon: 10.23333
Abee                           mass:     107000  lon: -113.0
Adzhi-Bogdo (stone)            mass:        910  lon: 95.16667
Acapulco                       mass:       1914  lon: -99.9
Achiras                        mass:        780  lon: -64.95
Adhi Kot                       mass:       4239  lon: 71.8
Aachen                         mass:         21  lon: 6.08333
Aïr                            mass:      24000  lon: 8.38333
Agen                           mass:      30000  lon: 0.61667
Aguada                         mass:       1620  lon: -65.23333
Akyumak                        mass:      50000  lon: 42.81667
Aioun el Atrouss               mass:       1000  lon: -9.57028
Aire-sur-la-Lys                mass:             lon: 2.33333
Akbarpur                       mass:       1800  lon: 77.95
Akaba                          mass:        779  lon: 35.05
Alby sur Chéran                mass:        252  lon: 6.01533
Alberta      

## Traversal — Look Up by Property Value

In [5]:
match = next(
    (f for f in data["features"] if f["properties"].get("name") == "Acapulco"),
    None
)

if match:
    print(match["properties"])
    print(match["geometry"]["coordinates"])

{'name': 'Acapulco', 'mass': 1914, 'year': 1976, 'id': 10, 'fall': 'Fell', 'recclass': 'Acapulcoite', 'marker-color': '#16ab1e', 'marker-size': 'small'}
[-99.9, 16.88333]


## Traversal — Collect One Field Across All Features

In [6]:
features = data["features"]

# all names
names = [f["properties"]["name"] for f in features]

# all (lon, lat) pairs
points = [f["geometry"]["coordinates"] for f in features]

# all masses — using .get() with a default since a field might be missing
masses = [f["properties"].get("mass", 0) for f in features]

print(names[:5])
print(points[:5])
print(masses[:5])

['Aarhus', 'Abee', 'Adzhi-Bogdo (stone)', 'Acapulco', 'Achiras']
[[10.23333, 56.18333], [-113.0, 54.21667], [95.16667, 44.83333], [-99.9, 16.88333], [-64.95, -33.16667]]
[720, 107000, 910, 1914, 780]


## GeoJSON vs Plain JSON — Key Differences

| | Plain JSON (`latex_colors.json`) | GeoJSON (`meteorites.geojson`) |
|---|---|---|
| Top-level type | array `[...]` | object `{...}` |
| How to get the list | `colors` directly | `data["features"]` |
| What each item is | an arbitrary object | always a `Feature` |
| Location data | none | `geometry.coordinates` |
| Attribute data | all keys at top level | lives inside `properties` |
| Coordinate order | n/a | `[lon, lat]` — not `[lat, lon]` |

The traversal pattern is the same — the structure just has a predictable shape every time.

## Exercise A

How many meteorites in this dataset have a recorded mass (non-`None`, non-zero)?

Use `.get("mass")` to safely access the field — some features may not have it.

In [15]:
features = data["features"]

# Count how many features have a non-None, non-zero mass value
# Your code here
sum_nonzeromasses = sum(1 for f in features if f["properties"].get("mass") != 0)
print(sum_nonzeromasses)

31945


## Exercise B

Find all meteorites that fell **after the year 2000**. Print their names and years.

Use `.get("year")` with a default since some entries may be missing that field.

In [16]:
features = data["features"]

# Find all meteorites that fell after year 2000
# Print their names and years
# Your code here

for f in features:
    year = f["properties"].get("year")
    if year and year > 2000:
        name = f["properties"].get("name")
        print(f"Name: {name}, Year: {year}")

Name: Alby sur Chéran, Year: 2002
Name: Al Zarnkh, Year: 2001
Name: Almahata Sitta, Year: 2008
Name: Ash Creek, Year: 2009
Name: Bassikounou, Year: 2006
Name: Battle Mountain, Year: 2012
Name: Benguerir, Year: 2004
Name: Beni M'hira, Year: 2001
Name: Bensour, Year: 2002
Name: Berduc, Year: 2008
Name: Berthoud, Year: 2004
Name: Bhawad, Year: 2002
Name: Boumdeid (2003), Year: 2003
Name: Boumdeid (2011), Year: 2011
Name: Bukhara, Year: 2001
Name: Bunburra Rockhole, Year: 2007
Name: Buzzard Coulee, Year: 2008
Name: Cali, Year: 2007
Name: Carancas, Year: 2007
Name: Chelyabinsk, Year: 2013
Name: Chergach, Year: 2007
Name: Daule, Year: 2008
Name: Devgaon, Year: 2001
Name: Dergaon, Year: 2001
Name: Didim, Year: 2007
Name: Grimsby, Year: 2009
Name: Hiroshima, Year: 2003
Name: Hoima, Year: 2003
Name: Huaxi, Year: 2010
Name: Jesenice, Year: 2009
Name: Jodiya, Year: 2006
Name: Kaprada, Year: 2004
Name: Kasauli, Year: 2003
Name: Kavarpura, Year: 2006
Name: Kemer, Year: 2008
Name: Kendrapara, Year: 

## Exercise C

Find the **3 northernmost** meteorites — those with the highest latitude. Print their names and latitudes, sorted from north to south.

Hint: latitude is `coordinates[1]`.

In [18]:
features = data["features"]

# Find the 3 northernmost meteorites — highest latitude (index 1 of coordinates)
# Sort and print their names and latitudes
# Your code here
for f in sorted(features, key=lambda f: f["geometry"]["coordinates"][1], reverse = True)[:3]:
    name = f["properties"].get("name")
    lat = f["geometry"]["coordinates"][1]
    print(f"Name: {name}, Latitude: {lat}")

Name: Ryder Gletcher, Latitude: 81.16667
Name: Thule, Latitude: 76.53333
Name: Cape York, Latitude: 76.13333




## Check Your Understanding

The meteorite `"Abee"` landed somewhere in Canada. Using the loaded `data` object, write code that prints its **latitude** and the **year** it fell.

```python
# your answer here

```


---

In [21]:
match = next(
    (f for f in data["features"] if f["properties"].get("name") == "Abee"),
    None
)

if match:
    lat = match["geometry"]["coordinates"][1]
    print(f"Abee, Latitude: {lat}")
   


Abee, Latitude: 54.21667


## Next

In [02 — Feature Collections](./02-Feature_Collections.ipynb), we go beyond reading and start building and modifying GeoJSON collections from scratch.